In [2]:
!pip install -q ultralytics opencv-python-headless requests tqdm

import os
import cv2
import glob
import json
import numpy as np
import requests
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [8]:
DATASET_ROOT = "/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103"  # adjust to your actual dataset slug

for root, dirs, files in os.walk(DATASET_ROOT):
    level = root.replace(DATASET_ROOT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"{indent}  {f}")

FoodSeg103/
  category_id.txt
  train_test_recipe1m_id.txt
  Readme.txt
  Images/
    img_dir/
      test/
      train/
    ann_dir/
      test/
      train/
  ImageSets/
    test.txt
    train.txt


In [9]:
DATASET_ROOT = "/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103"

OUT_LABELS_ROOT = "/kaggle/working/yolo_labels"

def mask_to_yolo_polygons(mask_path, img_w, img_h, min_area=50):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    class_ids = np.unique(mask)
    class_ids = class_ids[class_ids != 0]  # 0 = background

    lines = []
    for cls_id in class_ids:
        binary = (mask == cls_id).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < min_area:
                continue
            cnt = cnt.reshape(-1, 2)
            norm = []
            for x, y in cnt:
                norm.append(x / img_w)
                norm.append(y / img_h)
            # YOLO class ids are 0-indexed; FoodSeg103 masks are 1-indexed (0 = background)
            line = f"{int(cls_id) - 1} " + " ".join(f"{v:.6f}" for v in norm)
            lines.append(line)
    return lines

for split in ["train", "test"]:
    IMG_DIR = f"{DATASET_ROOT}/Images/img_dir/{split}"
    MASK_DIR = f"{DATASET_ROOT}/Images/ann_dir/{split}"
    OUT_LABELS = f"{OUT_LABELS_ROOT}/{split}"
    os.makedirs(OUT_LABELS, exist_ok=True)

    mask_files = glob.glob(f"{MASK_DIR}/*.png")
    print(f"Processing {split}: {len(mask_files)} masks found")

    for mask_path in tqdm(mask_files):
        fname = os.path.basename(mask_path).replace(".png", "")
        img_path = f"{IMG_DIR}/{fname}.jpg"
        if not os.path.exists(img_path):
            img_path = f"{IMG_DIR}/{fname}.png"
            if not os.path.exists(img_path):
                continue

        img = cv2.imread(img_path)
        h, w = img.shape[:2]

        lines = mask_to_yolo_polygons(mask_path, w, h)
        with open(f"{OUT_LABELS}/{fname}.txt", "w") as f:
            f.write("\n".join(lines))

print("Done converting masks to YOLO-Seg format for train and test.")

Processing train: 4983 masks found


100%|██████████| 4983/4983 [04:38<00:00, 17.87it/s]


Processing test: 2135 masks found


100%|██████████| 2135/2135 [01:04<00:00, 33.18it/s]

Done converting masks to YOLO-Seg format for train and test.


In [ ]:
import shutil

YOLO_ROOT = "/kaggle/working/foodseg_yolo"
for split in ["train", "val"]:
    os.makedirs(f"{YOLO_ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_ROOT}/labels/{split}", exist_ok=True)

# Symlink/copy images and labels into the expected structure
# (loop again per split — shown here for train; repeat for val)
for img_path in tqdm(glob.glob(f"{IMG_DIR}/*")):
    fname = os.path.basename(img_path)
    shutil.copy(img_path, f"{YOLO_ROOT}/images/train/{fname}")

for label_path in glob.glob(f"{OUT_LABELS}/*.txt"):
    fname = os.path.basename(label_path)
    shutil.copy(label_path, f"{YOLO_ROOT}/labels/train/{fname}")

# FoodSeg103 class names — replace with the actual category list from the dataset's category file
CLASS_NAMES = [f"class_{i}" for i in range(103)]  # placeholder, replace with real names

yaml_content = f"""
path: {YOLO_ROOT}
train: images/train
val: images/val
nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
with open(f"{YOLO_ROOT}/data.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

In [ ]:
model = YOLO("yolov8n-seg.pt")  # or yolov8s-seg.pt for better accuracy

results = model.train(
    data=f"{YOLO_ROOT}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/kaggle/working/runs",
    name="platecalc_seg"
)

best_model_path = "/kaggle/working/runs/platecalc_seg/weights/best.pt"

In [ ]:
def preprocess_image(image_path, target_size=640):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def load_image_for_inference(image_path):
    return preprocess_image(image_path)

In [ ]:
trained_model = YOLO(best_model_path)

def run_inference(image_path, conf_threshold=0.4):
    results = trained_model.predict(image_path, conf=conf_threshold)
    return results[0]

def extract_detections(result):
    detections = []
    if result.masks is None:
        return detections

    for i, mask in enumerate(result.masks.data):
        cls_id = int(result.boxes.cls[i].item())
        cls_name = result.names[cls_id]
        mask_np = mask.cpu().numpy().astype(np.uint8)
        area_px = int(np.sum(mask_np))
        detections.append({
            "class_id": cls_id,
            "class_name": cls_name,
            "mask": mask_np,
            "area_px": area_px
        })
    return detections

In [ ]:
# Foods counted as discrete instances vs. foods measured by area
COUNTABLE_FOODS = {"banana", "apple", "egg", "orange"}  # extend based on your class names

AVERAGE_WEIGHTS_G = {
    "banana": 118,
    "apple": 182,
    "egg": 50,
    "orange": 131,
}

def count_instances(mask):
    num_labels, labels = cv2.connectedComponents(mask)
    return num_labels - 1  # subtract background

def estimate_quantity(detection, image_area_px, plate_reference_area_g_per_px=None):
    cls_name = detection["class_name"]
    mask = detection["mask"]

    if cls_name in COUNTABLE_FOODS:
        instance_count = count_instances(mask)
        avg_weight = AVERAGE_WEIGHTS_G.get(cls_name, 100)  # default fallback
        return instance_count * avg_weight
    else:
        # area-based: proportion of image area, scaled to an assumed plate weight
        area_ratio = detection["area_px"] / image_area_px
        # Heuristic: assume full plate coverage of a food ~= 300g as a baseline
        estimated_weight = area_ratio * 300
        return estimated_weight

In [ ]:
USDA_API_KEY = "YOUR_API_KEY_HERE"

def get_calories_per_100g(food_name):
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {"api_key": USDA_API_KEY, "query": food_name, "pageSize": 1}
    response = requests.get(url, params=params)
    data = response.json()

    if not data.get("foods"):
        return None

    food = data["foods"][0]
    for nutrient in food.get("foodNutrients", []):
        if nutrient.get("nutrientName") == "Energy":
            return nutrient.get("value")  # kcal per 100g
    return None

# Cache lookups so you don't hit the API repeatedly for the same food
nutrition_cache = {}

def get_cached_calories(food_name):
    if food_name not in nutrition_cache:
        nutrition_cache[food_name] = get_calories_per_100g(food_name)
    return nutrition_cache[food_name]

In [ ]:
def compute_meal_calories(image_path):
    img = load_image_for_inference(image_path)
    image_area_px = img.shape[0] * img.shape[1]

    result = run_inference(image_path)
    detections = extract_detections(result)

    breakdown = []
    total_calories = 0

    for det in detections:
        weight_g = estimate_quantity(det, image_area_px)
        cal_per_100g = get_cached_calories(det["class_name"])

        if cal_per_100g is None:
            continue

        calories = (weight_g / 100) * cal_per_100g
        total_calories += calories

        breakdown.append({
            "food": det["class_name"],
            "estimated_weight_g": round(weight_g, 1),
            "calories_per_100g": cal_per_100g,
            "calories": round(calories, 1)
        })

    return {"breakdown": breakdown, "total_calories": round(total_calories, 1)}

In [ ]:
test_image = "/kaggle/input/your-test-image.jpg"  # replace with a real path

result = compute_meal_calories(test_image)
print(json.dumps(result, indent=2))

# Visualize
img = cv2.imread(test_image)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.title(f"Total: {result['total_calories']} kcal")
plt.axis("off")
plt.show()